# Testing `fit_similarity_alignment` (rotation-aware tissue-boundary fit)

`src/MERci/acquisition/alignment.py`'s existing `fit_isotropic_alignment` fits
scale + translation + optional axis-flip between two tissue-boundary
polygons, but **never a free rotation** -- fine for its original use
(planning where to re-image on a second scope, where the two stages are
assumed already axis-aligned), but not safe to assume for two experiments
imaged independently, each with its own arbitrary stage-insert angle.

This adds `fit_similarity_alignment` (same closed-form-then-IoU-refinement
approach, plus a coarse grid search over rotation angle) and is Stage A of
the cross-microscope cell-identity-mapping plan
(`prompt_history/2026_08_29_1637_plan_cross_microscope_cell_identity_mapping.md`).

**Part 1** recovers a known synthetic transform (sanity check on the fitting
code itself). **Part 2** applies it to the real `BC555_sample_05` `epi`/`disk`
tissue boundaries -- the actual Stage A coarse alignment the plan calls for.

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

MERCI_DIR = Path.cwd().parent.parent   # MERci/ (notebook lives in MERci/notebooks/tests/)
sys.path.insert(0, str(MERCI_DIR / "src"))

from shapely.geometry import Polygon

from MERci.acquisition.alignment import (
    load_boundary_polygon, fit_similarity_alignment, polygon_iou,
)

NOTEBOOK_NAME = "test_cross_microscope_similarity_alignment"
from MERci.visualization import get_merci_figures_dir

PLOT_TITLE_FONTSIZE    = 14
PLOT_LABEL_FONTSIZE    = 12
PLOT_TICK_FONTSIZE     = 11
PLOT_LEGEND_FONTSIZE   = 10
PLOT_SUPTITLE_FONTSIZE = 15


## Part 1 — recover a known synthetic transform

An irregular (non-symmetric) star-shaped polygon, so rotation actually changes the shape's orientation rather than mapping onto itself. The target is built by applying a *known* flip + rotation + scale + translation to the source, so the fit's recovered parameters have a ground truth to check against.

In [ ]:
rng = np.random.default_rng(0)
angle = np.linspace(0, 2 * np.pi, 15, endpoint=False)
r = 5 + 2 * np.sin(3 * angle) + rng.normal(0, 0.3, size=len(angle))
src_pts = np.c_[r * np.cos(angle), r * np.sin(angle)]
src_poly = Polygon(src_pts)

TRUE_FLIP_X, TRUE_FLIP_Y = True, False
TRUE_ROTATION_DEG = 110.0
TRUE_SCALE = 0.7
TRUE_TRANSLATION = np.array([-10.0, 200.0])

flipped = src_pts * np.array([-1.0 if TRUE_FLIP_X else 1.0, -1.0 if TRUE_FLIP_Y else 1.0])
theta = np.radians(TRUE_ROTATION_DEG)
R = np.array([[np.cos(theta), -np.sin(theta)], [np.sin(theta), np.cos(theta)]])
tgt_pts = (flipped @ R.T) * TRUE_SCALE + TRUE_TRANSLATION
tgt_poly = Polygon(tgt_pts)

fit = fit_similarity_alignment(src_poly, tgt_poly, n_angles=36)

print(f"flip_x       : fitted={fit.flip_x}   true={TRUE_FLIP_X}")
print(f"flip_y       : fitted={fit.flip_y}   true={TRUE_FLIP_Y}")
print(f"rotation_deg : fitted={fit.rotation_deg:.3f}   true={TRUE_ROTATION_DEG}")
print(f"scale        : fitted={fit.scale:.5f}   true={TRUE_SCALE}")
print(f"translation  : fitted=({fit.tx:.3f}, {fit.ty:.3f})   true={tuple(TRUE_TRANSLATION)}")
print(f"IoU          : {fit.iou:.6f}")

max_point_error = np.abs(fit.transform_points(src_pts) - tgt_pts).max()
print(f"\nmax |transform_points(src) - true target points| = {max_point_error:.2e}")

assert fit.flip_x == TRUE_FLIP_X and fit.flip_y == TRUE_FLIP_Y
assert abs(fit.rotation_deg - TRUE_ROTATION_DEG) < 0.5
assert abs(fit.scale - TRUE_SCALE) < 1e-3
assert max_point_error < 1e-3
assert fit.iou > 0.999
print("\nPASS -- fit_similarity_alignment recovers the known transform.")


## Part 2 — real `BC555_sample_05` epi/disk tissue boundaries

`epi` and `disk` are two separate experiment folders (not one shared `SAMPLE_DIR` with per-scope subfolders, unlike `align_fovs_across_microscopes.ipynb`'s assumed layout), so both directories are explicit here rather than auto-detected. Edit `EPI_DIR`/`DISK_DIR` for a different sample.

`disk` is the source (higher z-resolution, confocal) and `epi` the target, so the fitted transform maps disk coordinates into epi's frame -- the direction Stage C (segmentation cleanup) will need. `epi`'s boundary has 2 holes (`hole1.txt`, `hole2.txt`); this fits the outer ring only (see the plan doc's Stage A note) -- revisit if the fitted IoU below is surprisingly low.

In [ ]:
EPI_DIR  = Path("/n/holylfs05/LABS/zhuang_lab/Lab/shared/projects/breast_cancer/experiments/BC555_sample_05/epi")
DISK_DIR = Path("/n/holylfs05/LABS/zhuang_lab/Lab/shared/projects/breast_cancer/experiments/BC555_sample_05/disk")

EPI_BOUNDARY  = EPI_DIR  / "positions" / "boundaries" / "from_mosaic" / "boundary_positions.txt"
DISK_BOUNDARY = DISK_DIR / "positions" / "boundaries" / "from_mosaic" / "boundary_positions.txt"

for p in (EPI_BOUNDARY, DISK_BOUNDARY):
    print(f"{'OK ' if p.exists() else 'MISSING'}  {p}")


### Fit the transform (cached — per NOTEBOOK_GUIDELINES.md #2/#3)

Cached under `disk`'s own `analysis/cache/` (the source experiment), invalidated by either boundary file's mtime or the fit parameters changing.

In [ ]:
cache_dir  = DISK_DIR / "analysis" / "cache" / NOTEBOOK_NAME
cache_path = cache_dir / "epi_disk_boundary_fit.json"

N_ANGLES   = 72
ALLOW_FLIP = True

cache_key = {
    "epi_boundary_mtime":  EPI_BOUNDARY.stat().st_mtime,
    "disk_boundary_mtime": DISK_BOUNDARY.stat().st_mtime,
    "n_angles":            N_ANGLES,
    "allow_flip":          ALLOW_FLIP,
}

cached = None
if cache_path.exists():
    with open(cache_path) as fh:
        cached = json.load(fh)
    if cached.get("cache_key") != cache_key:
        cached = None

if cached is not None:
    print(f"Loaded cached fit: {cache_path}")
    fit_dict = cached["fit"]
else:
    epi_poly  = load_boundary_polygon(EPI_BOUNDARY)
    disk_poly = load_boundary_polygon(DISK_BOUNDARY)
    fit = fit_similarity_alignment(disk_poly, epi_poly, n_angles=N_ANGLES, allow_flip=ALLOW_FLIP)
    fit_dict = {
        "scale": fit.scale, "tx": fit.tx, "ty": fit.ty,
        "rotation_deg": fit.rotation_deg, "flip_x": fit.flip_x, "flip_y": fit.flip_y,
        "iou": fit.iou, "iou_init": fit.iou_init, "refined": fit.refined,
    }
    cache_dir.mkdir(parents=True, exist_ok=True)
    with open(cache_path, "w") as fh:
        json.dump({"cache_key": cache_key, "fit": fit_dict}, fh, indent=2)
    print(f"Saved fit: {cache_path}")

print()
for k, v in fit_dict.items():
    print(f"{k:12s}: {v}")


### Overlay plot

In [ ]:
from MERci.acquisition.alignment import AlignmentResult

fit = AlignmentResult(
    scale=fit_dict["scale"], tx=fit_dict["tx"], ty=fit_dict["ty"],
    iou=fit_dict["iou"], iou_init=fit_dict["iou_init"], n_iter=0,
    refined=fit_dict["refined"], flip_x=fit_dict["flip_x"], flip_y=fit_dict["flip_y"],
    rotation_deg=fit_dict["rotation_deg"],
)

epi_poly  = load_boundary_polygon(EPI_BOUNDARY)
disk_poly = load_boundary_polygon(DISK_BOUNDARY)
mapped_poly = fit.transform_polygon(disk_poly)


def _ring(poly):
    return np.asarray(poly.exterior.coords)


fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(13, 6))

ax0.plot(*_ring(disk_poly).T, "-", color="tab:blue",   label="disk boundary (source)")
ax0.plot(*_ring(epi_poly).T,  "-", color="tab:orange", label="epi boundary (target)")
ax0.set_title("Before: raw stage coordinates", fontsize=PLOT_TITLE_FONTSIZE)
ax0.tick_params(labelsize=PLOT_TICK_FONTSIZE)
ax0.legend(fontsize=PLOT_LEGEND_FONTSIZE, loc="best")
ax0.set_aspect("equal"); ax0.grid(alpha=0.3)

ax1.plot(*_ring(epi_poly).T,    "-",  color="tab:orange", lw=2, label="epi boundary (target)")
ax1.plot(*_ring(mapped_poly).T, "--", color="tab:blue",         label="disk→epi (fitted)")
ax1.set_title(f"After: IoU = {fit.iou:.3f}", fontsize=PLOT_TITLE_FONTSIZE)
ax1.tick_params(labelsize=PLOT_TICK_FONTSIZE)
ax1.legend(fontsize=PLOT_LEGEND_FONTSIZE, loc="best")
ax1.set_aspect("equal"); ax1.grid(alpha=0.3)

fig.suptitle(
    f"BC555_sample_05: disk→epi  (scale={fit.scale:.4f}, "
    f"rotation={fit.rotation_deg:.2f}°, flip_x={fit.flip_x}, flip_y={fit.flip_y})",
    fontsize=PLOT_SUPTITLE_FONTSIZE,
)
fig.tight_layout()

figures_dir = get_merci_figures_dir(DISK_DIR, "tests", NOTEBOOK_NAME)
figures_dir.mkdir(parents=True, exist_ok=True)
fig_path = figures_dir / f"{NOTEBOOK_NAME}.disk_to_epi_boundary_fit.png"
fig.savefig(fig_path, dpi=150)
print(f"Saved: {fig_path}")
plt.show()

if fit.iou < 0.8:
    print("\n⚠  Low IoU -- worth checking whether epi's holes need subtracting "
          "before fitting, or whether the two boundaries are genuinely a poor match.")


## Conclusion

`fit_similarity_alignment` recovers a known synthetic flip+rotation+scale+translation exactly (Part 1), and is backward-compatible with `fit_isotropic_alignment` (verified separately -- `rotation_deg=0.0` reduces `transform_points`/`transform_polygon` to the old formula exactly). Part 2's fitted disk→epi transform for `BC555_sample_05` is cached at `disk/analysis/cache/test_cross_microscope_similarity_alignment/epi_disk_boundary_fit.json` and is Stage A's real output -- the input Stage B (per-cell matching) will need once both experiments' MERlin segmentation finishes.